# Finanzas Públicas Estatales y Municipales (EFIPEM)

En este archivo se lee el monto de inversión pública de 2020 a 2025 por alcaldía de los reportes del INEGI que se pueden consultar en el siguiente enlace.

[Finanzas públicas](https://www.inegi.org.mx/sistemas/olap/consulta/general_ver4/MDXQueryDatos.asp?c=11291)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import os
import re
import tempfile
import shutil

CARPETA = "/content/drive/MyDrive/Seguridad"

Mounted at /content/drive


In [14]:
ARCHIVOS = [
    "INEGI_exporta_3_10_2026_16_41_8_2020.xls",
    "INEGI_exporta_3_10_2026_16_43_54_2021.xls",
    "INEGI_exporta_3_10_2026_17_1_14_2022.xls",
    "INEGI_exporta_3_10_2026_16_46_10_2023.xls",
    "INEGI_exporta_3_10_2026_17_11_9_2024.xls",
    "INEGI_exporta_3_10_2026_16_52_54_2025.xls"
]



In [15]:
def limpiar_monto(valor):

    if pd.isna(valor):
        return None
    texto = str(valor).strip()
    # Quitar espacios normales y especiales
    texto = texto.replace(" ", "").replace("\xa0", "")
    # Quitar comas
    texto = texto.replace(",", "")

    try:
        return float(texto)
    except:
        return None

In [16]:
def leer_excel_inegi(ruta):

    archivo_temporal = tempfile.NamedTemporaryFile(
        suffix=".xlsx",
        delete=False
    )

    archivo_temporal.close()

    shutil.copy(ruta, archivo_temporal.name)

    try:

        df = pd.read_excel(
            archivo_temporal.name,
            header=None,
            engine="openpyxl"
        )

    finally:

        os.remove(archivo_temporal.name)

    return df


In [17]:
# PROCESAR CADA ARCHIVO
def procesar_archivo(ruta):
    nombre = os.path.basename(ruta)
    print(f"\nProcesando: {nombre}")
    # Leer Excel completo sin asumir encabezados
    df_raw = leer_excel_inegi(ruta)
    # Encontrar el año DENTRO del archivo
    año = None

    for valor in df_raw.astype(str).values.flatten():

        match = re.search(
            r"Año\s*:\s*(\d{4})",
            str(valor),
            flags=re.IGNORECASE
        )

        if match:

            año = int(match.group(1))
            break


    print(f"   Año encontrado dentro del archivo: {año}")
    # Encontrar la fila que contiene "Inversión pública"


    fila_inversion = None

    for i in range(len(df_raw)):

        valores_fila = df_raw.iloc[i].astype(str)

        if valores_fila.str.contains(
            "Inversión pública",
            case=False,
            na=False
        ).any():

            fila_inversion = i
            break


    if fila_inversion is None:

        print("   ❌ No encontré la fila 'Inversión pública'")
        return None


    # La fila anterior contiene las alcaldías
    fila_alcaldias = fila_inversion - 1

    alcaldias = df_raw.iloc[fila_alcaldias].tolist()
    valores = df_raw.iloc[fila_inversion].tolist()

    # Construir tabla limpia
    datos = []

    for alcaldia, valor in zip(alcaldias, valores):
        if pd.isna(alcaldia):
            continue
        alcaldia = str(alcaldia).strip()

        if alcaldia == "" or alcaldia.lower() == "nan":
            continue
        monto = limpiar_monto(valor)
        if monto is not None:

            datos.append({

                "año": año,
                "alcaldia": alcaldia,
                "inversion_publica": monto,
                "archivo_origen": nombre

            })


    df_limpio = pd.DataFrame(datos)

    print(f"   ✓ Registros encontrados: {len(df_limpio)}")

    return df_limpio

dfs = []
for archivo in ARCHIVOS:
    ruta = os.path.join(CARPETA, archivo)
    # Verificar que exista
    if not os.path.exists(ruta):

        print(f"\n❌ NO ENCONTRADO: {archivo}")
        continue
    try:

        df_temp = procesar_archivo(ruta)

        if df_temp is not None and not df_temp.empty:

            dfs.append(df_temp)

    except Exception as e:

        print(f"❌ Error leyendo {archivo}")
        print(e)





Procesando: INEGI_exporta_3_10_2026_16_41_8_2020.xls
   Año encontrado dentro del archivo: 2020
   ✓ Registros encontrados: 17

Procesando: INEGI_exporta_3_10_2026_16_43_54_2021.xls
   Año encontrado dentro del archivo: 2021
   ✓ Registros encontrados: 17

Procesando: INEGI_exporta_3_10_2026_17_1_14_2022.xls
   Año encontrado dentro del archivo: 2022
   ✓ Registros encontrados: 17

Procesando: INEGI_exporta_3_10_2026_16_46_10_2023.xls
   Año encontrado dentro del archivo: 2023
   ✓ Registros encontrados: 17

Procesando: INEGI_exporta_3_10_2026_17_11_9_2024.xls
   Año encontrado dentro del archivo: 2024
   ✓ Registros encontrados: 17

Procesando: INEGI_exporta_3_10_2026_16_52_54_2025.xls
   Año encontrado dentro del archivo: 2025
   ✓ Registros encontrados: 17


In [18]:
# 7. UNIR TODO
df_inversion = pd.concat(
    dfs,
    ignore_index=True
)

# Quitar agregado de toda CDMX

df_inversion = df_inversion[
    df_inversion["alcaldia"].str.strip() != "Ciudad de México"
].copy()
#Ordenar
df_inversion = (
    df_inversion
    .sort_values(["año", "alcaldia"])
    .reset_index(drop=True)
)


# Convertir inversión a número entero nullable
df_inversion["inversion_publica"] = (
    df_inversion["inversion_publica"]
    .round()
    .astype("Int64")
)

In [19]:
print("\n" + "=" * 80)
print("DATAFRAME FINAL")
print("=" * 80)
display(df_inversion)
print("\nAños encontrados:")
print(sorted(df_inversion["año"].dropna().unique()))


print("\nShape:")
print(df_inversion.shape)


print("\nNúmero de alcaldías por año:")

display(
    df_inversion
    .groupby("año")
    .agg(
        alcaldias=("alcaldia", "nunique"),
        observaciones=("alcaldia", "size")
    )
    .reset_index()
)


DATAFRAME FINAL


,año,alcaldia,inversion_publica,archivo_origen
0,2020,Azcapotzalco,136744436,INEGI_exporta_3_10_2026_16_41_8_2020.xls
1,2020,Benito Juárez,216133458,INEGI_exporta_3_10_2026_16_41_8_2020.xls
2,2020,Coyoacán,302841069,INEGI_exporta_3_10_2026_16_41_8_2020.xls
3,2020,Cuajimalpa de Morelos,117699104,INEGI_exporta_3_10_2026_16_41_8_2020.xls
4,2020,Cuauhtémoc,207036359,INEGI_exporta_3_10_2026_16_41_8_2020.xls
...,...,...,...,...
91,2025,Tlalpan,427344937,INEGI_exporta_3_10_2026_16_52_54_2025.xls
92,2025,Tláhuac,217524129,INEGI_exporta_3_10_2026_16_52_54_2025.xls
93,2025,Venustiano Carranza,350208608,INEGI_exporta_3_10_2026_16_52_54_2025.xls
94,2025,Xochimilco,324746417,INEGI_exporta_3_10_2026_16_52_54_2025.xls



Años encontrados:
[np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

Shape:
(96, 4)

Número de alcaldías por año:


,año,alcaldias,observaciones
0,2020,16,16
1,2021,16,16
2,2022,16,16
3,2023,16,16
4,2024,16,16
5,2025,16,16


In [20]:
# Seleccionar las primeras 4 columnas
df_inversion = df_inversion.iloc[:, :3]

ruta_salida = "/content/drive/MyDrive/Seguridad/inversion_publica_alcaldia.csv"

# Guardar
df_inversion.to_csv(ruta_salida, index=False)


print("Archivo guardado: inversion_publica_alcaldia.csv")

Archivo guardado: inversion_publica_alcaldia.csv
